# Notebook 02 — Experiment 2: CDCC Thresholds and LLM Comprehension

Analyses the relationship between CDCC structural thresholds (cyclomatic complexity, LoC, nesting)  
and LLM comprehension quality (SCS, SAS) for a corpus of 100 Python functions.

**Prerequisites:** Run `make exp2` before executing this notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

from utils import EXP2_RESULTS, DATA_DIR

sns.set_theme(style='whitegrid')

metrics_df = pd.read_csv(DATA_DIR / 'code_metrics.csv')
scores_df  = pd.read_csv(EXP2_RESULTS)
merged = metrics_df.merge(scores_df, on='function_id', how='inner').sort_values('complexity')

print(f'Functions: {len(merged)}')
print(f'CDCC-compliant: {(~merged["cdcc_violation"]).sum()}')
merged[['complexity','loc','nesting_depth','scs','mean_sas','output_input_ratio']].describe().round(3)

## 2.1 Corpus Stratification

In [ ]:
bins = [0, 5, 10, 20, float('inf')]
labels = ['≤5 (T1)', '6–10 (T2)', '11–20 (T3)', '>20 (T4)']
merged['tier'] = pd.cut(merged['complexity'], bins=bins, labels=labels)

summary = merged.groupby('tier', observed=True)[['complexity','scs','mean_sas','output_input_ratio']].agg(['mean','std']).round(3)
summary

## 2.2 Figure 3 — SCS and SAS vs Cyclomatic Complexity

In [ ]:
CDCC_THRESHOLD = 10
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, label in [
    (axes[0], 'scs',      'Self-Consistency Score (SCS)'),
    (axes[1], 'mean_sas', 'Semantic Accuracy Score (SAS)'),
]:
    sub = merged[['complexity', col]].dropna()
    ax.scatter(sub['complexity'], sub[col], alpha=0.5, s=20, color='#2c3e50')
    rolling = sub.sort_values('complexity')[col].rolling(5, center=True, min_periods=1).mean()
    ax.plot(sub.sort_values('complexity')['complexity'], rolling, color='#e74c3c', lw=1.5, label='Rolling mean')
    ax.axvline(CDCC_THRESHOLD, color='#8e44ad', linestyle='--', lw=1.2, label=f'CDCC threshold ({CDCC_THRESHOLD})')
    ax.set_xlabel('Cyclomatic complexity')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend(fontsize=8)

fig.suptitle('Figure 3 — LLM comprehension quality vs cyclomatic complexity')
plt.tight_layout()
plt.show()

## 2.3 Figure 4 — Piecewise Linear Fit (Change-Point at CDCC Threshold)

In [ ]:
sub = merged[['complexity', 'scs']].dropna()
x, y = sub['complexity'].values, sub['scs'].values

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.4, s=18, color='#7f8c8d', label='Observations')

for mask, color, seg_label in [
    (x <= CDCC_THRESHOLD, '#2ecc71', 'Fit: complexity ≤ 10'),
    (x > CDCC_THRESHOLD,  '#e74c3c', 'Fit: complexity > 10'),
]:
    if mask.sum() >= 2:
        slope, intercept, *_ = linregress(x[mask], y[mask])
        xf = np.linspace(x[mask].min(), x[mask].max(), 100)
        ax.plot(xf, slope * xf + intercept, color=color, lw=2, label=seg_label)
        print(f'{seg_label}: slope={slope:.4f}')

ax.axvline(CDCC_THRESHOLD, color='#8e44ad', linestyle='--', lw=1.5, label='CDCC threshold')
ax.set_xlabel('Cyclomatic complexity')
ax.set_ylabel('Self-Consistency Score (SCS)')
ax.set_title('Figure 4 — Piecewise fit with CDCC change-point overlay')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2.4 Change-Point Detection (H3, H4)

In [ ]:
from changepoint_analysis import run as cp_run
results = cp_run()

for outcome, data in results.items():
    al = data['alignment']
    pw = data['piecewise']
    print(f'\n[{outcome}]')
    print(f'  Detected change-points : {al["detected_changepoints"]}')
    print(f'  Aligned to CDCC (tol=2): {al["aligned"]}')
    print(f'  Slope below cp=10      : {pw["slope_below"]:.4f}')
    print(f'  Slope above cp=10      : {pw["slope_above"]:.4f}')

## 2.5 Output/Input Token Ratio vs Complexity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sub = merged[['complexity', 'output_input_ratio']].dropna()
ax.scatter(sub['complexity'], sub['output_input_ratio'], alpha=0.4, s=18)
ax.axvline(CDCC_THRESHOLD, color='#8e44ad', linestyle='--', lw=1.2, label='CDCC threshold')
ax.set_xlabel('Cyclomatic complexity')
ax.set_ylabel('Output / Input token ratio')
ax.set_title('Output verbosity vs function complexity')
ax.legend()
plt.tight_layout()
plt.show()